In [3]:
import pandas as pd

# ---------------- Settings ----------------
FOUR_PT_LINE = 30
WEIGHTS = [3, 2, 1]        # last season, 2 seasons ago, 3 seasons ago
K_PCT = 50                 # regression to the mean: add 50 league-average attempts
TEST_SEASONS = ["2024-25", "2025-26"]

# ---------------- Load data: one row per player-season ----------------
shots = pd.read_csv("dataset2_top50_3pm_by_distance.csv")
shots["Distance"] = pd.to_numeric(shots["Shot Distance (ft)"].replace("40+", "40"))
summary = pd.read_csv("dataset2_top50_3pm_summary.csv")
keys = ["Season", "Player ID"]

deep = (shots[shots["Distance"] >= FOUR_PT_LINE].groupby(keys)[["3PM", "3PA"]].sum()
        .rename(columns={"3PM": "deep_M", "3PA": "deep_A"}))
data = summary[keys + ["Player", "GP"]].merge(deep, on=keys, how="left").fillna(0)
SEASONS = sorted(data["Season"].unique())


def project(target):
    """Project each player's extra PPG for `target` using up to 3 earlier seasons."""
    i = SEASONS.index(target)
    past = SEASONS[max(0, i - 3):i][::-1]                 # newest first
    league = data[data["Season"].isin(past)]
    league_pct = league["deep_M"].sum() / league["deep_A"].sum()

    rows = []
    for pid in data[data["Season"] == past[0]]["Player ID"].unique():
        gp = att = made = 0.0
        for w, season in zip(WEIGHTS, past):
            r = data[(data["Season"] == season) & (data["Player ID"] == pid)]
            if r.empty:
                continue                                  # not a top-50 shooter that season
            gp += w * r["GP"].iloc[0]
            att += w * r["deep_A"].iloc[0]
            made += w * r["deep_M"].iloc[0]
        attempts_per_game = att / gp                                  # weighted average
        make_pct = (made + K_PCT * league_pct) / (att + K_PCT)       # regressed to the mean
        rows.append({"Player ID": pid, "Projected": attempts_per_game * make_pct})
    return pd.DataFrame(rows)


def backtest(target):
    i = SEASONS.index(target)
    actual = data[data["Season"] == target].copy()
    actual["Actual"] = actual["deep_M"] / actual["GP"]
    last = data[data["Season"] == SEASONS[i - 1]].copy()
    last["Same as last season"] = last["deep_M"] / last["GP"]

    m = (project(target)
         .merge(actual[["Player ID", "Player", "Actual"]], on="Player ID")
         .merge(last[["Player ID", "Same as last season"]], on="Player ID"))
    m.insert(0, "Season", target)

    summary_row = {
        "Predicted season": target,
        "Players compared": len(m),
        "Model error": (m["Projected"] - m["Actual"]).abs().mean(),
        "Naive error": (m["Same as last season"] - m["Actual"]).abs().mean(),
        "Correlation": m["Projected"].corr(m["Actual"]),
        "Avg projected": m["Projected"].mean(),
        "Avg actual": m["Actual"].mean(),
    }
    return summary_row, m


results, details = zip(*[backtest(s) for s in TEST_SEASONS])
results = pd.DataFrame(results)
details = pd.concat(details, ignore_index=True)

pd.set_option("display.width", 140)
print("===== BACKTEST: extra PPG from 30+ ft makes =====")
print("Error = average miss in extra points per game (lower is better)\n")
print(results.round(3).to_string(index=False))

for season in TEST_SEASONS:
    print(f"\n===== {season}: biggest projected gainers vs. what happened =====")
    t = details[details["Season"] == season].sort_values("Projected", ascending=False).head(10)
    print(t[["Player", "Projected", "Actual", "Same as last season"]].round(2).to_string(index=False))

details.drop(columns="Player ID").to_csv("backtest_results.csv", index=False)
print("\nSaved every player's projected vs. actual numbers to backtest_results.csv")

===== BACKTEST: extra PPG from 30+ ft makes =====
Error = average miss in extra points per game (lower is better)

Predicted season  Players compared  Model error  Naive error  Correlation  Avg projected  Avg actual
         2024-25                32        0.049        0.049        0.800          0.085       0.103
         2025-26                25        0.039        0.051        0.823          0.085       0.085

===== 2024-25: biggest projected gainers vs. what happened =====
           Player  Projected  Actual  Same as last season
       Trae Young       0.43    0.42                 0.43
   Damian Lillard       0.40    0.24                 0.29
    Stephen Curry       0.27    0.44                 0.24
     Jordan Poole       0.19    0.47                 0.17
Tyrese Haliburton       0.18    0.14                 0.25
  Anfernee Simons       0.15    0.11                 0.20
      Luka Dončić       0.13    0.36                 0.14
 Donovan Mitchell       0.13    0.15                